# ch03_5. 심장질환 모델 학습 최적화

## 표준화·학습률·과적합·L2 규제 비교

## 개요

- Training Set(트레이닝 세트, 훈련 데이터), Validation Set(밸리데이션 세트, 검증 데이터), Test Set(테스트 세트, 테스트 데이터)의 역할을 구분합니다.
- 검증 데이터로 모델을 선택하고 테스트 데이터는 마지막에 한 번만 사용합니다.

## 1. 실습 흐름

```text
Heart 데이터 불러오기
→ Heart 데이터 전처리
→ 훈련·검증·테스트 분리
→ 기준 선형 모델
→ 표준화 적용 여부 비교
→ 학습률 비교
→ 복잡한 신경망의 과적합 관찰
→ L2 규제 적용
→ 검증 결과로 모델 선택
→ 테스트 데이터 최종 평가
```

| 실험 | 변경 조건 | 관찰 항목 |
|---|---|---|
| 기준 모델 | 표준화 + 선형 모델 | 비교 기준 |
| 표준화 제거 | 입력 크기만 변경 | 초기 손실과 학습 안정성 |
| 학습률 비교 | 학습률만 변경 | 느린 학습과 빠른 학습 |
| 복잡한 모델 | 은닉층 추가 | 훈련·검증 손실 차이 |
| L2 규제 | `weight_decay` 적용 | 과적합 완화 |

## 2. 라이브러리와 CSV 파일 준비

`heart.csv`를 사용합니다.

In [ ]:
from pathlib import Path
import copy

import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

data_path = Path("heart.csv")

if not data_path.exists():
    data_path = Path("heart(1).csv")

if not data_path.exists():
    try:
        from google.colab import files
        files.upload()

        data_path = Path("heart.csv")

        if not data_path.exists():
            data_path = Path("heart(1).csv")

    except ImportError:
        pass

if not data_path.exists():
    raise FileNotFoundError(
        "현재 폴더에 heart.csv 또는 heart(1).csv를 넣어 주세요."
    )

heart_data = pd.read_csv(data_path)

print("사용 파일:", data_path.name)
print("데이터 shape:", heart_data.shape)
print("결측값 수:", heart_data.isna().sum().sum())
print()
print("target 분포:")
print(heart_data["target"].value_counts().sort_index())

## 3. 데이터 전처리

- 연속형 특성: `age`, `trestbps`, `chol`, `thalach`, `oldpeak`
- 이진형 특성: `sex`, `fbs`, `exang`
- 다범주형 특성: `cp`, `restecg`, `slope`, `ca`, `thal`
- 다범주형 특성만 One-Hot Encoding(원핫 인코딩) 적용

In [ ]:
continuous_columns = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak",
]

categorical_columns = [
    "cp",
    "restecg",
    "slope",
    "ca",
    "thal",
]

X = heart_data.drop(
    columns="target"
).copy()

y = heart_data[["target"]].astype(
    np.float32
)

X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    dtype=np.float32,
).astype(np.float32)

print("원본 입력 특성 수:", X.shape[1])
print("원핫 인코딩 후 입력 특성 수:", X_encoded.shape[1])

## 4. 훈련·검증·테스트 데이터 분리

데이터를 `60% : 20% : 20%`로 나눕니다.

```text
훈련 데이터
→ 가중치 학습

검증 데이터
→ 학습률·모델 크기·규제 비교

테스트 데이터
→ 최종 선택된 모델 평가
```

`train_test_split()`을 두 번 사용하면 복잡한 사용자 정의 분할 함수 없이 세 집합을 만들 수 있습니다.

In [ ]:
# 1차 분할: 임시 데이터 80%, 테스트 데이터 20%
X_temp, X_test, y_temp, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y,
)

# 2차 분할: 전체 기준 훈련 60%, 검증 20%
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_temp,
)

print("훈련 데이터:", X_train.shape, y_train.shape)
print("검증 데이터:", X_val.shape, y_val.shape)
print("테스트 데이터:", X_test.shape, y_test.shape)

print()
print("target=1 비율")
print("훈련:", round(y_train["target"].mean(), 4))
print("검증:", round(y_val["target"].mean(), 4))
print("테스트:", round(y_test["target"].mean(), 4))

## 5. 표준화 데이터와 원본 데이터 준비

연속형 특성만 훈련 데이터의 평균과 표준편차로 표준화합니다.

$$
x_{\mathrm{scaled}}
=
\frac{x-\mu_{\mathrm{train}}}
{\sigma_{\mathrm{train}}}
$$

표준화 효과를 비교하기 위해 다음 두 입력을 모두 준비합니다.

- `standardized`: 연속형 특성을 표준화한 데이터
- `raw`: 원래 크기를 유지한 데이터

In [ ]:
train_mean = X_train[
    continuous_columns
].mean()

train_std = X_train[
    continuous_columns
].std().replace(0, 1)

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled.loc[:, continuous_columns] = (
    X_train[continuous_columns] - train_mean
) / train_std

X_val_scaled.loc[:, continuous_columns] = (
    X_val[continuous_columns] - train_mean
) / train_std

X_test_scaled.loc[:, continuous_columns] = (
    X_test[continuous_columns] - train_mean
) / train_std

print("표준화 후 훈련 평균:")
print(
    X_train_scaled[continuous_columns]
    .mean()
    .round(3)
)

print()
print("표준화 후 훈련 표준편차:")
print(
    X_train_scaled[continuous_columns]
    .std()
    .round(3)
)

## 6. Tensor로 변환

복잡한 중첩 딕셔너리는 사용하지 않고 필요한 Tensor를 각각 만듭니다.

In [ ]:
def feature_tensor(dataframe):
    return torch.tensor(
        dataframe.to_numpy(),
        dtype=torch.float32,
    )


def target_tensor(dataframe):
    return torch.tensor(
        dataframe.to_numpy(),
        dtype=torch.float32,
    )


train_x = feature_tensor(X_train_scaled)
val_x = feature_tensor(X_val_scaled)
test_x = feature_tensor(X_test_scaled)

raw_train_x = feature_tensor(X_train)
raw_val_x = feature_tensor(X_val)

train_y = target_tensor(y_train)
val_y = target_tensor(y_val)
test_y = target_tensor(y_test)

input_size = train_x.shape[1]

print("입력 특성 수:", input_size)
print("훈련 Tensor:", train_x.shape, train_y.shape)
print("검증 Tensor:", val_x.shape, val_y.shape)
print("테스트 Tensor:", test_x.shape, test_y.shape)

## 7. 공통 학습 함수

실험마다 학습 반복문을 복사하지 않도록 하나의 함수를 사용합니다.

함수의 역할:

1. 모델 학습
2. 매 Epoch의 훈련·검증 손실 기록
3. 검증 손실이 가장 낮았던 Epoch 저장
4. 가장 좋은 가중치 복원

코드는 길어 보이지만, 이후 실험 코드를 짧고 동일하게 유지하기 위한 공통 부분입니다.

In [ ]:
loss_function = nn.BCEWithLogitsLoss()


def accuracy_from_logits(logits, targets):
    predictions = (
        torch.sigmoid(logits) >= 0.5
    ).float()

    return (
        predictions == targets
    ).float().mean().item()


def train_model(
    model,
    train_features,
    validation_features,
    learning_rate=0.01,
    epochs=600,
    weight_decay=0.0,
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": [],
    }

    best_val_loss = float("inf")
    best_epoch = 0
    best_state = copy.deepcopy(
        model.state_dict()
    )

    for epoch in range(epochs):
        model.train()

        train_logits = model(
            train_features
        )

        train_loss = loss_function(
            train_logits,
            train_y,
        )

        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        model.eval()

        with torch.no_grad():
            train_logits = model(
                train_features
            )

            val_logits = model(
                validation_features
            )

            current_train_loss = loss_function(
                train_logits,
                train_y,
            ).item()

            current_val_loss = loss_function(
                val_logits,
                val_y,
            ).item()

            current_train_accuracy = (
                accuracy_from_logits(
                    train_logits,
                    train_y,
                )
            )

            current_val_accuracy = (
                accuracy_from_logits(
                    val_logits,
                    val_y,
                )
            )

        history["train_loss"].append(
            current_train_loss
        )

        history["val_loss"].append(
            current_val_loss
        )

        history["train_accuracy"].append(
            current_train_accuracy
        )

        history["val_accuracy"].append(
            current_val_accuracy
        )

        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            best_epoch = epoch + 1

            best_state = copy.deepcopy(
                model.state_dict()
            )

    # 검증 손실이 가장 낮았던 가중치 복원
    model.load_state_dict(best_state)

    return {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
    }

## 8. 실험 1: 기준 선형 모델

기준 조건:

- 표준화 적용
- `nn.Linear(입력 특성 수, 1)`
- Adam(아담, 적응형 최적화 알고리즘)
- 학습률 `0.01`
- 600 Epoch

In [ ]:
torch.manual_seed(RANDOM_SEED)

baseline_model = nn.Linear(
    input_size,
    1,
)

baseline_result = train_model(
    model=baseline_model,
    train_features=train_x,
    validation_features=val_x,
    learning_rate=0.01,
    epochs=600,
)

print(
    "최저 검증 손실 Epoch:",
    baseline_result["best_epoch"],
)

print(
    "최저 검증 손실:",
    round(
        baseline_result["best_val_loss"],
        4,
    ),
)

print(
    "마지막 훈련 정확도:",
    round(
        baseline_result["history"][
            "train_accuracy"
        ][-1],
        4,
    ),
)

print(
    "마지막 검증 정확도:",
    round(
        baseline_result["history"][
            "val_accuracy"
        ][-1],
        4,
    ),
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    baseline_result["history"]["train_loss"],
    label="Train Loss",
)

plt.plot(
    baseline_result["history"]["val_loss"],
    label="Validation Loss",
)

plt.axvline(
    baseline_result["best_epoch"] - 1,
    linestyle="--",
    label="Best Epoch",
)

plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Baseline Model")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 9. 실험 2: 표준화 제거

모델·학습률·Epoch는 유지하고 입력의 표준화만 제거합니다.

비교할 항목:

- 첫 번째 Epoch의 훈련 손실
- 검증 손실 감소 속도
- 최저 검증 손실

> 표준화가 항상 정확도를 높인다고 단정하지 않습니다.
> 초기 손실과 학습 안정성까지 함께 비교합니다.

In [ ]:
torch.manual_seed(RANDOM_SEED)

raw_model = nn.Linear(
    input_size,
    1,
)

raw_result = train_model(
    model=raw_model,
    train_features=raw_train_x,
    validation_features=raw_val_x,
    learning_rate=0.01,
    epochs=600,
)

print(
    "표준화 적용 첫 훈련 손실:",
    round(
        baseline_result["history"][
            "train_loss"
        ][0],
        4,
    ),
)

print(
    "표준화 미적용 첫 훈련 손실:",
    round(
        raw_result["history"][
            "train_loss"
        ][0],
        4,
    ),
)

print()
print(
    "표준화 적용 최저 검증 손실:",
    round(
        baseline_result["best_val_loss"],
        4,
    ),
)

print(
    "표준화 미적용 최저 검증 손실:",
    round(
        raw_result["best_val_loss"],
        4,
    ),
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    baseline_result["history"]["val_loss"],
    label="Standardized",
)

plt.plot(
    raw_result["history"]["val_loss"],
    label="Raw Scale",
)

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Effect of Standardization")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 10. 실험 3: 학습률 비교

표준화된 데이터와 선형 모델을 고정하고 학습률만 바꿉니다.

| 학습률 | 예상되는 특징 |
|---:|---|
| `0.0001` | 제한된 Epoch에서 학습이 매우 느림 |
| `0.01` | 기준 학습률 |
| `0.1` | 빠르게 학습하지만 최저점을 지나칠 수 있음 |

한 번에 조건 하나만 바꾸어야 결과 변화의 원인을 해석하기 쉽습니다.

In [ ]:
learning_rates = [
    0.0001,
    0.01,
    0.1,
]

learning_rate_results = {}

for learning_rate in learning_rates:
    torch.manual_seed(RANDOM_SEED)

    model = nn.Linear(
        input_size,
        1,
    )

    result = train_model(
        model=model,
        train_features=train_x,
        validation_features=val_x,
        learning_rate=learning_rate,
        epochs=400,
    )

    learning_rate_results[
        learning_rate
    ] = result


learning_rate_rows = []

for learning_rate, result in (
    learning_rate_results.items()
):
    learning_rate_rows.append(
        {
            "learning_rate": learning_rate,
            "best_epoch": result["best_epoch"],
            "best_val_loss": result[
                "best_val_loss"
            ],
            "last_val_loss": result[
                "history"
            ]["val_loss"][-1],
            "last_val_accuracy": result[
                "history"
            ]["val_accuracy"][-1],
        }
    )

learning_rate_summary = pd.DataFrame(
    learning_rate_rows
)

print(
    learning_rate_summary
    .round(4)
    .to_string(index=False)
)

In [ ]:
plt.figure(figsize=(8, 5))

for learning_rate in learning_rates:
    result = learning_rate_results[
        learning_rate
    ]

    plt.plot(
        result["history"]["val_loss"],
        label=f"lr={learning_rate}",
    )

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss by Learning Rate")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 11. 실험 4: 복잡한 모델과 과적합

복잡한 모델은 은닉층 두 개만 사용해 구조를 단순하게 유지합니다.

```text
입력 27개
→ Linear(27, 32)
→ ReLU
→ Linear(32, 16)
→ ReLU
→ Linear(16, 1)
```

과적합의 대표적인 흐름:

```text
훈련 손실 계속 감소
+
검증 손실 다시 증가
→ 훈련 데이터에 지나치게 맞춰질 가능성
```

In [ ]:
def make_complex_model():
    return nn.Sequential(
        nn.Linear(input_size, 32),
        nn.ReLU(),
        nn.Linear(32, 16),
        nn.ReLU(),
        nn.Linear(16, 1),
    )


torch.manual_seed(RANDOM_SEED)

complex_result = train_model(
    model=make_complex_model(),
    train_features=train_x,
    validation_features=val_x,
    learning_rate=0.01,
    epochs=1200,
    weight_decay=0.0,
)

print(
    "최저 검증 손실 Epoch:",
    complex_result["best_epoch"],
)

print(
    "최저 검증 손실:",
    round(
        complex_result["best_val_loss"],
        4,
    ),
)

print(
    "마지막 훈련 손실:",
    round(
        complex_result["history"][
            "train_loss"
        ][-1],
        4,
    ),
)

print(
    "마지막 검증 손실:",
    round(
        complex_result["history"][
            "val_loss"
        ][-1],
        4,
    ),
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    complex_result["history"]["train_loss"],
    label="Train Loss",
)

plt.plot(
    complex_result["history"]["val_loss"],
    label="Validation Loss",
)

plt.axvline(
    complex_result["best_epoch"] - 1,
    linestyle="--",
    label="Best Epoch",
)

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Overfitting Observation")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 12. 실험 5: L2 규제

L2 규제는 가중치가 지나치게 커지는 것을 억제합니다.

$$
\mathrm{Total\ Loss}
=
\mathrm{Data\ Loss}
+
\lambda
\sum_i w_i^2
$$

PyTorch에서는 Adam의 `weight_decay`로 간단히 적용할 수 있습니다.

차이를 쉽게 관찰하기 위해 `weight_decay=0.01`을 사용합니다.

In [ ]:
torch.manual_seed(RANDOM_SEED)

regularized_result = train_model(
    model=make_complex_model(),
    train_features=train_x,
    validation_features=val_x,
    learning_rate=0.01,
    epochs=1200,
    weight_decay=0.01,
)

print(
    "L2 적용 최저 검증 손실 Epoch:",
    regularized_result["best_epoch"],
)

print(
    "L2 적용 최저 검증 손실:",
    round(
        regularized_result["best_val_loss"],
        4,
    ),
)

print(
    "L2 적용 마지막 검증 손실:",
    round(
        regularized_result["history"][
            "val_loss"
        ][-1],
        4,
    ),
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    complex_result["history"]["val_loss"],
    label="Complex Model",
)

plt.plot(
    regularized_result["history"][
        "val_loss"
    ],
    label="Complex Model + L2",
)

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Effect of L2 Regularization")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 13. 검증 데이터로 모델 선택

테스트 데이터는 아직 사용하지 않습니다.

비교 대상:

- 기준 선형 모델
- 표준화하지 않은 선형 모델
- 복잡한 모델
- 복잡한 모델 + L2

검증 손실이 가장 낮은 모델을 최종 모델로 선택합니다.

In [ ]:
experiment_results = {
    "Baseline Linear": baseline_result,
    "Raw Scale Linear": raw_result,
    "Complex Model": complex_result,
    "Complex Model + L2": regularized_result,
}

validation_rows = []

for name, result in experiment_results.items():
    validation_rows.append(
        {
            "experiment": name,
            "best_epoch": result["best_epoch"],
            "learning_rate": result[
                "learning_rate"
            ],
            "weight_decay": result[
                "weight_decay"
            ],
            "best_val_loss": result[
                "best_val_loss"
            ],
        }
    )

validation_summary = pd.DataFrame(
    validation_rows
).sort_values(
    "best_val_loss"
)

print(
    validation_summary
    .round(4)
    .to_string(index=False)
)

selected_name = validation_summary.iloc[
    0
]["experiment"]

selected_result = experiment_results[
    selected_name
]

selected_model = selected_result["model"]

print()
print("선택된 모델:", selected_name)

## 14. 선택된 모델의 테스트 최종 평가

이제 테스트 데이터를 처음 사용합니다.

평가 지표:

- Accuracy(애큐러시, 정확도)
- Precision(프리시전, 정밀도)
- Recall(리콜, 재현율)
- F1 Score(에프원 스코어)
- Confusion Matrix(컨퓨전 매트릭스, 혼동행렬)

In [ ]:
selected_model.eval()

with torch.no_grad():
    test_logits = selected_model(
        test_x
    )

    test_probabilities = torch.sigmoid(
        test_logits
    )

    test_predictions = (
        test_probabilities >= 0.5
    ).float()


actual = test_y.reshape(-1)
predicted = test_predictions.reshape(-1)

tp = int(
    ((actual == 1) & (predicted == 1)).sum()
)

tn = int(
    ((actual == 0) & (predicted == 0)).sum()
)

fp = int(
    ((actual == 0) & (predicted == 1)).sum()
)

fn = int(
    ((actual == 1) & (predicted == 0)).sum()
)

accuracy = (
    tp + tn
) / len(actual)

precision = tp / max(
    tp + fp,
    1,
)

recall = tp / max(
    tp + fn,
    1,
)

f1 = (
    2 * precision * recall
    / max(precision + recall, 1e-12)
)

print("선택된 모델:", selected_name)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print()
print("TN:", tn, "FP:", fp)
print("FN:", fn, "TP:", tp)

In [ ]:
confusion_matrix = np.array(
    [
        [tn, fp],
        [fn, tp],
    ]
)

plt.figure(figsize=(5, 4))
plt.imshow(confusion_matrix)

plt.xticks(
    [0, 1],
    ["Predicted 0", "Predicted 1"],
)

plt.yticks(
    [0, 1],
    ["Actual 0", "Actual 1"],
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Final Test Confusion Matrix")

for row in range(2):
    for column in range(2):
        plt.text(
            column,
            row,
            confusion_matrix[row, column],
            ha="center",
            va="center",
            fontsize=14,
        )

plt.colorbar()
plt.tight_layout()
plt.show()

## 16. 핵심 정리

- 훈련 데이터는 가중치 학습에 사용합니다.
- 훈련 손실이 감소해도 검증 손실이 증가하면 과적합 가능성이 있습니다.
- 최종 모델은 훈련 정확도가 아니라 검증 성능을 기준으로 선택합니다.